In [1]:
# 1. Import library
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

import pickle
import matplotlib.pyplot as plt

In [2]:

data_dir = "../dataset/food-20"  # path dataset
img_height, img_width = 224, 224
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/train",
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir + "/val",
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names

# 3. Dataset
img_height, img_width = 180, 180
batch_size = 32
data_dir = "../dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Optimize dataset (prefetching)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 4. Base model MobileNetV2 (transfer learning)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze weights

# 5. Add custom classifier on top
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax')
])

# 6. Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 7. Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10  # bisa ditambah kalau GPU kuat
)

# 8. Cek performa
acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f"Training Accuracy: {acc:.2f}, Validation Accuracy: {val_acc:.2f}")

# 9. Save model
model.save("mobilenetv2.h5")
print("✅ Model MobileNetV2 saved as mobilenetv2.h5")

Found 1400 files belonging to 20 classes.
Found 400 files belonging to 20 classes.
Found 2000 files belonging to 1 classes.
Using 1600 files for training.
Found 2000 files belonging to 1 classes.
Using 400 files for validation.


C:\Users\kangg\AppData\Local\Temp\ipykernel_13356\4267751094.py:48: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 36s 536ms/step - accuracy: 0.7360 - loss: 0.9642 - val_accuracy: 1.0000 - val_loss: 4.3501e-05
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 504ms/step - accuracy: 1.0000 - loss: 6.8683e-05 - val_accuracy: 1.0000 - val_loss: 3.7713e-05
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 506ms/step - accuracy: 1.0000 - loss: 6.5942e-05 - val_accuracy: 1.0000 - val_loss: 3.7071e-05
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 507ms/step - accuracy: 1.0000 - loss: 6.8342e-05 - val_accuracy: 1.0000 - val_loss: 3.6352e-05
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 514ms/step - accuracy: 1.0000 - loss: 6.3593e-05 - val_accuracy: 1.0000 - val_loss: 3.5563e-05
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 25s 509ms/step - accuracy: 1.0000 - loss: 6.7722e-05 - val_accuracy: 1.0000 - val_loss: 3.4706e-05
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 24s 485ms/step - accuracy: 1.0000 - loss: 6.9854e-05 - val_accuracy: 1.0000 - val_loss: 3.3789e-05
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 524

Training Accuracy: 1.00, Validation Accuracy: 1.00
✅ Model MobileNetV2 saved as mobilenetv2.h5


In [3]:
print("Jumlah class:", len(class_names))

count_train = sum(1 for _ in train_ds.unbatch())
count_val = sum(1 for _ in val_ds.unbatch())

print("Jumlah gambar training:", count_train)
print("Jumlah gambar validation:", count_val)
print("Class names:", class_names)


Jumlah class: 20
Jumlah gambar training: 1600
Jumlah gambar validation: 400
Class names: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'breakfast_burrito', 'caesar_salad', 'cheesecake', 'chicken_curry', 'churros', 'donuts', 'edamame', 'falafel', 'french_fries', 'frozen_yogurt', 'grilled_cheese_sandwich', 'hamburger', 'hot_and_sour_soup', 'ice_cream', 'lasagna']
